In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto.')

for candidate in [
    PROJECT_ROOT / 'code',
    PROJECT_ROOT / 'code' / 'tmdb',
]:
    if candidate.exists() and str(candidate.resolve()) not in sys.path:
        sys.path.append(str(candidate.resolve()))

from IPython.display import display

from tmdb_api_utils import resolve_tmdb_api_key
from tmdb_feature_enrichment_utils import (
    collect_tmdb_additional_metadata,
    load_movies_for_tmdb_enrichment,
)

/home/gabriel/Faculdade/Matérias/ML/UFSJ_Aprendizado_Maquina_TP1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Enriquecimento Adicional com Metadados do TMDB**

Este notebook fica responsável apenas pela **coleta incremental** dos novos metadados do TMDB. A ideia aqui é gerar e manter o cache bruto com `release_date`, `director_name`, `top_billed_cast`, `production_company_names` e variáveis agregadas ligadas à produção e ao período de estreia.

A análise exploratória e o pré-processamento dessas novas colunas ficam concentrados no notebook `04`, para manter a mesma separação de responsabilidades usada no restante do repositório.

In [2]:
PROCESSED_DATA_INPUT = PROJECT_ROOT / 'data' / 'TMDB_movies_processed.csv'
TMDB_METADATA_OUTPUT = PROJECT_ROOT / 'data' / 'TMDB_movies_additional_metadata.csv'

TMDB_API_KEY_FALLBACK = None
TMDB_API_KEY_ENV = 'TMDB_API_KEY'
TMDB_SLEEP_SECONDS = 0.25
SHOW_PROGRESS = True
MAX_MOVIES = None
OVERWRITE_CACHES = False

TOP_CAST_MEMBERS = 3

## **Campos coletados no cache bruto**

Para cada `id_tmdb`, o notebook consulta os detalhes do filme e os créditos no TMDB e salva:

- `release_date`, `release_year`, `release_month`, `release_quarter` e `release_period`;
- `director_name`;
- `top_billed_cast`, limitado aos primeiros nomes do elenco configurados em `TOP_CAST_MEMBERS`;
- `production_company_names` e `main_production_company`;
- `production_company_count`, `cast_member_count`, `is_summer_release` e `is_holiday_release`.

O resultado desta etapa é o arquivo `data/TMDB_movies_additional_metadata.csv`.

In [3]:
df_movies = load_movies_for_tmdb_enrichment(PROCESSED_DATA_INPUT)
print(df_movies.shape)
display(df_movies.head())

(6918, 73)


,id_tmdb,title,runtime,adult,belongs_to_collection,budget,Action,Adventure,Animation,Comedy,...,sv,ta,te,th,tn,tr,uk,ur,zh,revenue
0,552524,Lilo & Stitch,108,0,0,100000000,0,0,0,1,...,0,0,0,0,0,0,0,0,0,610800000
1,950387,A Minecraft Movie,101,0,1,150000000,0,1,0,1,...,0,0,0,0,0,0,0,0,0,947000000
2,1257960,सिकंदर,133,0,0,23500000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,24727058
3,574475,Final Destination Bloodlines,110,0,1,50000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,229314062
4,1197306,A Working Man,116,0,0,40000000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,98652557


In [4]:
api_key = resolve_tmdb_api_key(TMDB_API_KEY_FALLBACK, TMDB_API_KEY_ENV)

tmdb_metadata_df = collect_tmdb_additional_metadata(
    df_movies,
    api_key=api_key,
    output_path=TMDB_METADATA_OUTPUT,
    pause_seconds=TMDB_SLEEP_SECONDS,
    overwrite_existing=OVERWRITE_CACHES,
    max_movies=MAX_MOVIES,
    top_cast_members=TOP_CAST_MEMBERS,
    show_progress=SHOW_PROGRESS,
)

display(tmdb_metadata_df.head())
tmdb_metadata_df['tmdb_fetch_status'].value_counts(dropna=False)

TMDB metadata enrichment: 100%|██████████| 6918/6918 [47:20<00:00,  2.44it/s]


,id_tmdb,title,tmdb_title,release_date,release_year,release_month,release_quarter,release_period,is_summer_release,is_holiday_release,director_name,top_billed_cast,production_company_names,main_production_company,production_company_count,cast_member_count,tmdb_fetch_status,tmdb_http_status,tmdb_error
0,5,Four Rooms,Four Rooms,1995-12-09,1995,12,Q4,verao,0,1,Allison Anders,Tim Roth|Jennifer Beals|David Proval,Miramax|A Band Apart,Miramax,2,28,ok,200,
1,6,Judgment Night,Judgment Night,1993-10-15,1993,10,Q4,primavera,0,0,Stephen Hopkins,Emilio Estevez|Cuba Gooding Jr.|Denis Leary,Largo Entertainment|JVC|Universal Pictures,Largo Entertainment,3,31,ok,200,
2,11,Star Wars,Star Wars,1977-05-25,1977,5,Q2,outono,1,0,George Lucas,Mark Hamill|Harrison Ford|Carrie Fisher,Lucasfilm Ltd.,Lucasfilm Ltd.,1,104,ok,200,
3,12,Finding Nemo,Finding Nemo,2003-05-30,2003,5,Q2,outono,1,0,Andrew Stanton,Albert Brooks|Ellen DeGeneres|Alexander Gould,Pixar,Pixar,1,68,ok,200,
4,13,Forrest Gump,Forrest Gump,1994-06-23,1994,6,Q2,inverno,1,0,Robert Zemeckis,Tom Hanks|Robin Wright|Gary Sinise,Paramount Pictures|The Steve Tisch Company|Wen...,Paramount Pictures,3,146,ok,200,


tmdb_fetch_status
ok    6918
Name: count, dtype: int64

In [5]:
selected_columns = [
    'id_tmdb',
    'title',
    'release_date',
    'director_name',
    'top_billed_cast',
    'production_company_names',
    'production_company_count',
    'cast_member_count',
    'tmdb_fetch_status',
]

display(tmdb_metadata_df[selected_columns].head(10))
print(f'Cache bruto salvo em: {TMDB_METADATA_OUTPUT}')
print(f'Total de filmes no cache: {tmdb_metadata_df.shape[0]}')

,id_tmdb,title,release_date,director_name,top_billed_cast,production_company_names,production_company_count,cast_member_count,tmdb_fetch_status
0,5,Four Rooms,1995-12-09,Allison Anders,Tim Roth|Jennifer Beals|David Proval,Miramax|A Band Apart,2,28,ok
1,6,Judgment Night,1993-10-15,Stephen Hopkins,Emilio Estevez|Cuba Gooding Jr.|Denis Leary,Largo Entertainment|JVC|Universal Pictures,3,31,ok
2,11,Star Wars,1977-05-25,George Lucas,Mark Hamill|Harrison Ford|Carrie Fisher,Lucasfilm Ltd.,1,104,ok
3,12,Finding Nemo,2003-05-30,Andrew Stanton,Albert Brooks|Ellen DeGeneres|Alexander Gould,Pixar,1,68,ok
4,13,Forrest Gump,1994-06-23,Robert Zemeckis,Tom Hanks|Robin Wright|Gary Sinise,Paramount Pictures|The Steve Tisch Company|Wen...,3,146,ok
5,14,American Beauty,1999-09-15,Sam Mendes,Kevin Spacey|Annette Bening|Thora Birch,DreamWorks Pictures|Jinks/Cohen Company,2,41,ok
6,15,Citizen Kane,1941-04-17,Orson Welles,Orson Welles|Joseph Cotten|Dorothy Comingore,Mercury Productions|RKO Radio Pictures,2,154,ok
7,16,Dancer in the Dark,2000-09-01,Lars von Trier,Björk|Catherine Deneuve|David Morse,Zentropa Entertainments|DR|SVT Drama|ARTE|Fran...,15,69,ok
8,18,Le Cinquième Élément,1997-05-02,Luc Besson,Bruce Willis|Milla Jovovich|Gary Oldman,Gaumont,1,120,ok
9,19,Metropolis,1927-01-10,Fritz Lang,Gustav Fröhlich|Brigitte Helm|Alfred Abel,UFA,1,24,ok


Cache bruto salvo em: /home/gabriel/Faculdade/Matérias/ML/UFSJ_Aprendizado_Maquina_TP1/data/TMDB_movies_additional_metadata.csv
Total de filmes no cache: 6918
